# Lab 7: Compression — Pruning and Distillation (Kaggle T4 GPU Runner)

**Hardware Acceleration:** NVIDIA Tesla T4 GPU (Kaggle Notebooks)
**Course:** Machine Learning & Efficient AI
**Author:** Muhammadjon Negmatov

---

### Overview of Laboratory Objectives
This Kaggle notebook resumes and completes the remaining experiments for Lab 7 according to Chapter 9 (*Efficient AI*) and Chapter 10 (*Model Optimizations*) by Reddi et al.:

1. **Task 1 (Reused):** Oversized Baseline CNN (~1M parameters, teacher model).
2. **Task 2A (Reused):** Global Unstructured Magnitude Pruning across sparsity targets: `[0.0, 0.25, 0.50, 0.75, 0.90, 0.95]`.
3. **Task 2B (Resumed):** Layer-wise Unstructured Magnitude Pruning: inspects verified completed rows (e.g. 0% and 25%) and executes remaining targets (50%, 75%, 90%, 95%) with 3-epoch fixed-mask fine-tuning.
4. **Task 2C (Executed):** Structured Filter Pruning (L2 norm) across ratios: `[10%, 25%, 50%]`, contracting physical model dimensions.
5. **Task 2D & 3 (Executed):** Latency and Storage Analysis (Dense vs Compressed vs Sparse COO) and comparison plots.
6. **Task 4 (Executed):** Knowledge Distillation using Teacher (1M baseline) to Student (~100k parameters) with $\alpha = 0.5$ and $T = 4.0$:
   $$\mathcal{L} = \alpha \cdot \mathcal{L}_{\text{CE}}(\mathbf{z}_s, \mathbf{y}) + (1 - \alpha) \cdot T^2 \cdot \mathcal{D}_{\text{KL}}\left(\text{softmax}\left(\frac{\mathbf{z}_s}{T}\right) \parallel \text{softmax}\left(\frac{\mathbf{z}_t}{T}\right)\right)$$
7. **Task 5 (Executed):** Four Pillars Efficiency Analysis (Data, Model, Compute, Energy).
8. **Packaging:** Saves all completed results, checkpoints, and plots into `/kaggle/working/lab07_completed_results.zip` for download.

In [ ]:
# Cell 1: Check CUDA and Tesla T4 GPU Availability
import torch
import sys
import os

print("=== Kaggle Hardware Environment Check ===")
cuda_available = torch.cuda.is_available()
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Accelerator: {gpu_name}")
    print(f"Available VRAM: {vram_gb:.2f} GB")
    !nvidia-smi
else:
    print("WARNING: GPU is not active! In Kaggle, click Settings (right pane) -> Accelerator -> GPU T4 x1.")


In [ ]:
# Cell 2: Install Missing Dependencies (Without replacing PyTorch or CUDA)
!pip install -q --no-warn-conflicts thop torchinfo pyyaml
print("Required packages verified!")


In [ ]:
# Cell 3: Discover and Setup Lab 7 in /kaggle/working/lab07
import os
import shutil
import zipfile
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working").resolve()
TARGET_LAB07 = KAGGLE_WORKING / "lab07"

# Search for uploaded lab07 dataset under /kaggle/input/
input_dir = Path("/kaggle/input")
found_source = None

if input_dir.exists():
    for candidate in input_dir.rglob("*"):
        if candidate.is_file() and candidate.name in ["lab07.zip", "lab07_archive.zip"]:
            found_source = candidate
            break
        elif candidate.is_dir() and candidate.name == "lab07" and (candidate / "checkpoints").exists():
            found_source = candidate
            break

print(f"Detected source dataset: {found_source}")

if not TARGET_LAB07.exists():
    if found_source and found_source.is_file() and found_source.suffix == ".zip":
        print(f"Extracting {found_source} to /kaggle/working/...")
        with zipfile.ZipFile(found_source, "r") as zf:
            zf.extractall(KAGGLE_WORKING)
    elif found_source and found_source.is_dir():
        print(f"Copying {found_source} to {TARGET_LAB07}...")
        shutil.copytree(found_source, TARGET_LAB07, dirs_exist_ok=True)
    else:
        print("No attached dataset found in /kaggle/input/. Creating empty lab07 structure...")
        TARGET_LAB07.mkdir(parents=True, exist_ok=True)

print(f"Project ready in: {TARGET_LAB07}")
print(f"Contents of {TARGET_LAB07}:")
for p in TARGET_LAB07.iterdir():
    print(f"  {p.name}")


In [ ]:
# Cell 4: Verify Existing Checkpoints and Completed CSV Results
import sys
import pandas as pd

if str(KAGGLE_WORKING) not in sys.path:
    sys.path.insert(0, str(KAGGLE_WORKING))

ckpt_path = TARGET_LAB07 / "checkpoints" / "baseline_teacher.pt"
global_csv = TARGET_LAB07 / "results" / "global_pruning_results.csv"
layer_csv = TARGET_LAB07 / "results" / "layerwise_pruning_results.csv"

print("=== Inspecting Existing Experiment Artifacts ===")
if ckpt_path.exists():
    print(f"[OK] Baseline teacher checkpoint verified: {ckpt_path} ({ckpt_path.stat().st_size / (1024*1024):.2f} MB)")
else:
    print(f"[MISSING] Baseline teacher checkpoint not found at {ckpt_path}")

if global_csv.exists():
    gdf = pd.read_csv(global_csv)
    print(f"[OK] Global pruning results verified ({len(gdf)} targets): {gdf['target_sparsity'].tolist()}")
else:
    print(f"[MISSING] Global pruning CSV not found at {global_csv}")

if layer_csv.exists():
    ldf = pd.read_csv(layer_csv)
    valid_targets = []
    for _, row in ldf.iterrows():
        if pd.notna(row.get("post_finetune_test_acc")) and float(row.get("post_finetune_test_acc")) > 0:
            valid_targets.append(round(float(row["target_sparsity"]), 4))
    print(f"[VERIFIED] Layer-wise completed targets: {valid_targets}")
else:
    print(f"[PENDING] Layer-wise pruning CSV not yet generated at {layer_csv}")


In [ ]:
# Cell 5: Execute Task 2 & Task 3 (Pruning + Latency & Storage Benchmarking)
# Resumes verified completed targets and executes remaining ones:
# - Skips verified Global Pruning (0% to 95%)
# - Skips verified Layer-wise Pruning (0% and 25%)
# - Runs Layer-wise Pruning: 50%, 75%, 90%, 95%
# - Runs Structured Filter Pruning: 10%, 25%, 50%
# - Measures dense vs sparse on-disk storage and CPU latency
# - Generates publication-ready plots under /kaggle/working/lab07/plots/

%cd /kaggle/working
!PYTHONPATH=/kaggle/working python3 -u lab07/scripts/run_pruning_experiments.py


In [ ]:
# Cell 6: Execute Task 4 (Knowledge Distillation)
# Reuses frozen Baseline teacher (baseline_teacher.pt)
# 1. Trains Plain Student (CE Only, 8 epochs)
# 2. Trains Distilled Student (Hinton KD, T=4.0, alpha=0.5, 8 epochs)
# Profiles accuracy, latency, FLOPs, and generates learning curves

%cd /kaggle/working
!PYTHONPATH=/kaggle/working python3 -u lab07/scripts/run_distillation_experiments.py


In [ ]:
# Cell 7: Execute Task 5 (Four Pillars Efficiency Synthesis)
# Compiles Data, Model, Compute, and Energy metrics across all models
# Outputs four_pillars_efficiency.csv and four_pillars_table.md

%cd /kaggle/working
!PYTHONPATH=/kaggle/working python3 -u lab07/scripts/run_efficiency_analysis.py


In [ ]:
# Cell 8: Run Complete Lab 7 Automated Test Suite
%cd /kaggle/working
!pytest lab07/tests/ -v


In [ ]:
# Cell 9: Package All Completed Results and Artifacts into ZIP Archive
import zipfile
import os
from pathlib import Path

zip_out = Path("/kaggle/working/lab07_completed_results.zip")
folders_to_include = [
    "lab07/results",
    "lab07/plots",
    "lab07/checkpoints",
    "lab07/reports",
    "lab07/configs",
]

print(f"Creating {zip_out}...")
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    for rel_folder in folders_to_include:
        abs_folder = KAGGLE_WORKING / rel_folder
        if abs_folder.exists():
            for root, _, files in os.walk(abs_folder):
                for f in files:
                    full_p = Path(root) / f
                    arcname = str(full_p.relative_to(KAGGLE_WORKING))
                    zf.write(full_p, arcname)

print(f"Packaging complete! File saved at: {zip_out} ({zip_out.stat().st_size / (1024*1024):.2f} MB)")
print("You can now download this file directly from the Kaggle Output pane on the right!")
